# AtomFlow Controller — Unified PYNQ Control

Combines image analysis (reconstruct) + atom sorting into the single
`atomflow_controller` IP running on the FPGA.

Two modes:
- **MODE_QUBIT_READOUT (0)**: image analysis only → read emissions
- **MODE_INITIALIZATION (1)**: image analysis + sorting → read emissions + moves from AXI-Stream FIFO

Test configuration matches `tb_atomflow_controller.cpp` exactly.

## 1. Imports and Overlay

In [ ]:
from pynq import Overlay, allocate, MMIO
import numpy as np
import struct
import time
import matplotlib.pyplot as plt

print("Loading overlay...")
ol = Overlay("atomflow_system_wrapper.bit")
ol.download()
print("\u2713 Overlay loaded")
print("IP dict:", list(ol.ip_dict.keys()))

## 2. Register Map

Offsets from `xatomflow_controller_hw.h` (Vitis HLS 2024.2).
Scalars and m_axi address registers are **interleaved** — do not use old offset tables.

In [ ]:
# ap_ctrl
CTRL            = 0x00
AP_START        = 1 << 0
AP_DONE         = 1 << 1
AP_IDLE         = 1 << 2
AP_READY        = 1 << 3

# Scalars and m_axi pointers (interleaved, from xatomflow_controller_hw.h)
MODE_OFFSET          = 0x10   # uint8_t  mode
THRESHOLD_OFFSET     = 0x18   # float    emission_threshold
ATOM_LOC_SIZE        = 0x20   # int      atomLocationsSize
PROJ_SHAPE0          = 0x28   # int      projShape0
PROJ_SHAPE1          = 0x30   # int      projShape1
ADDR_ATOM_LOC_LO     = 0x38   # m_axi atomLocations [31:0]
ADDR_ATOM_LOC_HI     = 0x3C   # m_axi atomLocations [63:32]
PSF_SUPER            = 0x44   # int      psfSupersample
IMG_PROJ_SIZE        = 0x4C   # int      imageProjectionSize
ADDR_PROJS_LOCAL_LO  = 0x54   # m_axi imageProjs_local [31:0]
ADDR_PROJS_LOCAL_HI  = 0x58
ADDR_IMAGE_PROJS_LO  = 0x60   # m_axi imageProjs [31:0]
ADDR_IMAGE_PROJS_HI  = 0x64
ADDR_PROJS_LOCAL_SZ_LO = 0x6C # m_axi imageProjs_local_size [31:0]
ADDR_PROJS_LOCAL_SZ_HI = 0x70
ADDR_FULL_IMAGE_LO   = 0x78   # m_axi fullImage [31:0]
ADDR_FULL_IMAGE_HI   = 0x7C
FULL_IMG_ROWS        = 0x84   # int      fullImage_rows
FULL_IMG_COLS        = 0x8C   # int      fullImage_cols
ADDR_EMISSIONS_LO    = 0x94   # m_axi emissions [31:0]
ADDR_EMISSIONS_HI    = 0x98
GRID_ROWS            = 0xA0   # int      grid_rows
GRID_COLS            = 0xA8   # int      grid_cols
ADDR_TARGET_GEO_LO   = 0xB0   # m_axi targetGeometry_mem [31:0]
ADDR_TARGET_GEO_HI   = 0xB4
COMP_ROW_START       = 0xBC   # uint     compZoneRowStart
COMP_ROW_END         = 0xC4   # uint     compZoneRowEnd
COMP_COL_START       = 0xCC   # uint     compZoneColStart
COMP_COL_END         = 0xD4   # uint     compZoneColEnd
MOVE_COUNT_REG       = 0xDC   # uint     moveCount (read-only)

MODE_QUBIT_READOUT  = 0
MODE_INITIALIZATION = 1

# AXI4-Stream FIFO (axi_fifo_mm_s)
FIFO_RDFR = 0x18   # Receive FIFO Reset (write 0xA5)
FIFO_RDFO = 0x1C   # Receive FIFO Occupancy (word count)
FIFO_RDFD = 0x20   # Receive FIFO Data
FIFO_RLR  = 0x24   # Receive Length Register (byte count)

CTRL_BASE = ol.ip_dict["atomflow_controller_0"]["phys_addr"]
FIFO_BASE = ol.ip_dict["axi_fifo_mm_s_0"]["phys_addr"]

ctrl = MMIO(CTRL_BASE, 0x1000)
fifo = MMIO(FIFO_BASE, 0x1000)

print(f"atomflow_controller @ {hex(CTRL_BASE)}")
print(f"AXI-Stream FIFO     @ {hex(FIFO_BASE)}")
v = ctrl.read(CTRL)
print(f"AP_CTRL = 0x{v:08X}  (IDLE={bool(v & AP_IDLE)})")

## 3. Helper Functions

In [ ]:
def write_addr64(mmio, lo_off, hi_off, addr):
    addr = int(addr)
    mmio.write(lo_off, addr & 0xFFFFFFFF)
    mmio.write(hi_off, (addr >> 32) & 0xFFFFFFFF)

def write_float(mmio, offset, val):
    bits = struct.unpack("<I", struct.pack("<f", float(val)))[0]
    mmio.write(offset, bits)

def atomflow_run(mode, emission_threshold,
                 atom_loc_size, proj_shape0, proj_shape1,
                 psf_supersample, img_proj_size,
                 full_img_rows, full_img_cols,
                 grid_rows, grid_cols,
                 comp_row_start, comp_row_end, comp_col_start, comp_col_end,
                 atomLocations_buf, imageProjs_buf, imageProjs_local_buf,
                 imageProjs_local_size_buf, fullImage_buf,
                 emissions_buf, targetGeometry_buf,
                 timeout_s=30.0):
    """Configure and run atomflow_controller. Returns (ms, move_count)."""
    # Write in hw.h address order (scalars and m_axi pointers interleaved)
    ctrl.write(MODE_OFFSET,    mode)
    write_float(ctrl, THRESHOLD_OFFSET, emission_threshold)
    ctrl.write(ATOM_LOC_SIZE,  atom_loc_size)
    ctrl.write(PROJ_SHAPE0,    proj_shape0)
    ctrl.write(PROJ_SHAPE1,    proj_shape1)
    write_addr64(ctrl, ADDR_ATOM_LOC_LO,       ADDR_ATOM_LOC_HI,       atomLocations_buf.physical_address)
    ctrl.write(PSF_SUPER,      psf_supersample)
    ctrl.write(IMG_PROJ_SIZE,  img_proj_size)
    write_addr64(ctrl, ADDR_PROJS_LOCAL_LO,    ADDR_PROJS_LOCAL_HI,    imageProjs_local_buf.physical_address)
    write_addr64(ctrl, ADDR_IMAGE_PROJS_LO,    ADDR_IMAGE_PROJS_HI,    imageProjs_buf.physical_address)
    write_addr64(ctrl, ADDR_PROJS_LOCAL_SZ_LO, ADDR_PROJS_LOCAL_SZ_HI, imageProjs_local_size_buf.physical_address)
    write_addr64(ctrl, ADDR_FULL_IMAGE_LO,     ADDR_FULL_IMAGE_HI,     fullImage_buf.physical_address)
    ctrl.write(FULL_IMG_ROWS,  full_img_rows)
    ctrl.write(FULL_IMG_COLS,  full_img_cols)
    write_addr64(ctrl, ADDR_EMISSIONS_LO,      ADDR_EMISSIONS_HI,      emissions_buf.physical_address)
    ctrl.write(GRID_ROWS,      grid_rows)
    ctrl.write(GRID_COLS,      grid_cols)
    write_addr64(ctrl, ADDR_TARGET_GEO_LO,     ADDR_TARGET_GEO_HI,     targetGeometry_buf.physical_address)
    ctrl.write(COMP_ROW_START, comp_row_start)
    ctrl.write(COMP_ROW_END,   comp_row_end)
    ctrl.write(COMP_COL_START, comp_col_start)
    ctrl.write(COMP_COL_END,   comp_col_end)

    # Quick sanity check
    rb_mode = ctrl.read(MODE_OFFSET) & 0xFF
    rb_zrs  = ctrl.read(COMP_ROW_START)
    rb_zre  = ctrl.read(COMP_ROW_END)
    print(f"  [check] mode={rb_mode} compZoneRows=[{rb_zrs},{rb_zre})")
    if rb_mode != mode or rb_zrs != comp_row_start or rb_zre != comp_row_end:
        print("  WARNING: register readback mismatch!")

    for _ in range(10000):
        if ctrl.read(CTRL) & AP_IDLE:
            break
        time.sleep(0.001)
    else:
        raise TimeoutError("IP never went IDLE")

    t0 = time.perf_counter()
    ctrl.write(CTRL, AP_START)

    while True:
        if ctrl.read(CTRL) & AP_DONE:
            break
        if time.perf_counter() - t0 > timeout_s:
            raise TimeoutError(f"Timeout after {timeout_s}s")
        time.sleep(0.001)

    elapsed_ms = (time.perf_counter() - t0) * 1e3
    move_count = ctrl.read(MOVE_COUNT_REG)
    return elapsed_ms, move_count

## 4. Allocate Memory Buffers

Sizes from HLS header (image_analysis.hpp):
- `IMAGE_PROJECTION_LOCAL = 1000`  (floats per atom in local proj buffer)
- `IMAGE_PROJECTION_SIZE  = 100`   (max atoms)
- `FULL_IMAGE_SIZE = 1024×1024`

In [ ]:
# HLS constants (from atomflow_controller.hpp / image_analysis.hpp)
MAX_ATOM_SITES   = 2000         # MAX_ATOM_SITES in atomflow_controller.hpp
IMAGE_PROJ_SIZE  = 100          # IMAGE_PROJECTION_SIZE (max projections)
IMAGE_PROJ_LOCAL = 1000         # IMAGE_PROJECTION_LOCAL (floats per atom in local buf)
FULL_IMAGE_SIZE  = 1024 * 1024  # PIXEL * PIXEL

# atomLocations and emissions: indexed by atom site (up to 2000)
# imageProjs*: indexed by projection (up to 100)
atomLocations_buf         = allocate(shape=(MAX_ATOM_SITES * 2,),               dtype=np.float32)
imageProjs_buf            = allocate(shape=(IMAGE_PROJ_SIZE,),                  dtype=np.float32)
imageProjs_local_buf      = allocate(shape=(IMAGE_PROJ_LOCAL * IMAGE_PROJ_SIZE,), dtype=np.float32)
imageProjs_local_size_buf = allocate(shape=(IMAGE_PROJ_SIZE,),                  dtype=np.int32)
fullImage_buf             = allocate(shape=(FULL_IMAGE_SIZE,),                  dtype=np.float32)
emissions_buf             = allocate(shape=(MAX_ATOM_SITES,),                   dtype=np.float32)
targetGeometry_buf        = allocate(shape=(64 * 64,),                          dtype=np.uint8)

print(f"atomLocations_buf         @ {hex(atomLocations_buf.physical_address)}")
print(f"imageProjs_buf            @ {hex(imageProjs_buf.physical_address)}")
print(f"imageProjs_local_buf      @ {hex(imageProjs_local_buf.physical_address)}")
print(f"imageProjs_local_size_buf @ {hex(imageProjs_local_size_buf.physical_address)}")
print(f"fullImage_buf             @ {hex(fullImage_buf.physical_address)}")
print(f"emissions_buf             @ {hex(emissions_buf.physical_address)}")
print(f"targetGeometry_buf        @ {hex(targetGeometry_buf.physical_address)}")

## 5. Load Input Data

Uses the same test files as `tb_atomflow_controller.cpp`:
- `restoutput.txt` (atom locations, projections)
- `fullImage_output.txt` (camera image)

`input_read.py` matches testbench parsing exactly:
- atom_location struct = `{float x; float y;}` → file order is y first, x second
- fullImage re-strided to PIXEL=1024 column layout before packing

In [ ]:
import sys, os
sys.path.insert(0, ".")
from input_read import parse_input_file, parse_fullImage, pack_floats_to_512bit, pack_fullimage_512bit, IMAGE_PROJECTION_LOCAL

# ---- adjust paths to where files are on the board ----
INPUT_FILE    = "restoutput.txt"
FULL_IMG_FILE = "fullImage_output.txt"
# -------------------------------------------------------

t0 = time.time()
input_data, _     = parse_input_file(INPUT_FILE)
full_image_raw, _ = parse_fullImage(FULL_IMG_FILE)
print(f"Parsed in {time.time()-t0:.3f}s")

atom_count  = input_data["atomLocationsSize"]
proj_shape0 = input_data["projShape0"]
proj_shape1 = input_data["projShape1"]
psf_super   = input_data["psfSupersample"]
img_proj_sz = input_data["imageProjectionSize"]
full_rows   = input_data["fullImage_rows"]
full_cols   = input_data["fullImage_cols"]

print(f"  atom_count  : {atom_count}")
print(f"  img_proj_sz : {img_proj_sz}")
print(f"  proj_shape  : {proj_shape0} x {proj_shape1}")
print(f"  full_image  : {full_rows} x {full_cols}")
assert atom_count  <= MAX_ATOM_SITES,  f"atom_count {atom_count} > MAX_ATOM_SITES {MAX_ATOM_SITES}"
assert img_proj_sz <= IMAGE_PROJ_SIZE, f"img_proj_sz {img_proj_sz} > IMAGE_PROJ_SIZE {IMAGE_PROJ_SIZE}"

# Atom locations: struct {float x; float y;}, testbench: y=parts[0], x=parts[1]
atomLocations_buf[:] = 0.0
for i, (x, y) in enumerate(input_data["atomLocations"][:atom_count]):
    atomLocations_buf[2*i]   = np.float32(x)
    atomLocations_buf[2*i+1] = np.float32(y)

# imageProjs (one float per projection, up to IMAGE_PROJ_SIZE)
n_projs = min(img_proj_sz, IMAGE_PROJ_SIZE)
imageProjs_buf[:n_projs] = np.array(input_data["imageProjs"][:n_projs], dtype=np.float32)

# imageProjs_local (stride IMAGE_PROJECTION_LOCAL=1000 per projection)
local_flat = np.array(input_data["imageProjs_local"], dtype=np.float32)
n_local = min(len(local_flat), IMAGE_PROJ_LOCAL * IMAGE_PROJ_SIZE)
imageProjs_local_buf[:n_local] = local_flat[:n_local]

# imageProjs_local_size
local_sizes = input_data.get("imageProjs_local_size", [])
if not local_sizes:
    local_sizes = [proj_shape0 * proj_shape1] * n_projs
imageProjs_local_size_buf[:n_projs] = np.array(local_sizes[:n_projs], dtype=np.int32)

# Re-stride to PIXEL=1024 column layout (testbench: fullImg_strided[r*PIXEL+c] = fullImg[r*fcols+c])
full_image_strided = pack_fullimage_512bit(full_image_raw, full_rows, full_cols)
fullImage_buf[:FULL_IMAGE_SIZE] = full_image_strided[:FULL_IMAGE_SIZE]

for buf in [atomLocations_buf, imageProjs_buf, imageProjs_local_buf,
            imageProjs_local_size_buf, fullImage_buf]:
    buf.flush()
print("\u2713 Buffers loaded and flushed")

In [ ]:
# --- Debug: verify buffer contents after loading ---
print(f"full_image_raw   : {len(full_image_raw)} values, first={full_image_raw[:3]}")
print(f"fullImage_buf    : max={np.max(fullImage_buf):.1f}  nonzero={np.count_nonzero(fullImage_buf)}")
print(f"imageProjs_buf   : max={np.max(imageProjs_buf[:n_projs]):.1f}  n_projs={n_projs}")
print(f"imageProjs_local : max={np.max(imageProjs_local_buf):.1f}  nonzero={np.count_nonzero(imageProjs_local_buf)}")
print(f"atomLoc[0]       : x={atomLocations_buf[0]:.2f}  y={atomLocations_buf[1]:.2f}")
print(f"img_proj_sz={img_proj_sz}, atom_count={atom_count}, full_rows={full_rows}, full_cols={full_cols}")

## 6. Configure Grid and Target Geometry

In [ ]:
# Grid size
GRID_R = 16
GRID_C = 16

# Computation zone = full grid (matches testbench: ZRS=0, ZRE=16, ZCS=0, ZCE=16)
COMP_R0, COMP_R1 = 0, GRID_R
COMP_C0, COMP_C1 = 0, GRID_C

def make_target(rows, cols):
    """Testbench target pattern:
    rows r%4==0 or r%4==1: cols 1..(cols//2-2) and (cols//2+1)..(cols-2) filled.
    For 16 cols: cols 1-6 and 9-14 are filled."""
    tgt = np.zeros(rows * cols, dtype=np.uint8)
    half = cols // 2
    for r in range(rows):
        if r % 4 == 0 or r % 4 == 1:
            for c in range(cols):
                if (1 <= c <= half - 2) or (half + 1 <= c <= cols - 2):
                    tgt[r * cols + c] = 1
    return tgt

target_flat = make_target(GRID_R, GRID_C)
target_2d   = target_flat.reshape(GRID_R, GRID_C)

np.copyto(targetGeometry_buf[:GRID_R * GRID_C], target_flat)
targetGeometry_buf.flush()

print(f"Target: {np.sum(target_flat)} sites in {GRID_R}x{GRID_C} grid")
print(f"Comp zone: rows [{COMP_R0},{COMP_R1}) x cols [{COMP_C0},{COMP_C1})")
print("Target pattern:")
for r in range(GRID_R):
    row_str = "".join("X" if target_2d[r,c] else "." for c in range(GRID_C))
    print(f"  {row_str}")

## 7a. Run — MODE_QUBIT_READOUT

Runs image analysis only → compute emissions, derive adaptive threshold.

In [ ]:
print("Running MODE_QUBIT_READOUT...")
elapsed_ms, move_count = atomflow_run(
    mode=MODE_QUBIT_READOUT,
    emission_threshold=0.0,
    atom_loc_size=atom_count,
    proj_shape0=proj_shape0, proj_shape1=proj_shape1,
    psf_supersample=psf_super, img_proj_size=img_proj_sz,
    full_img_rows=full_rows, full_img_cols=full_cols,
    grid_rows=GRID_R, grid_cols=GRID_C,
    comp_row_start=COMP_R0, comp_row_end=COMP_R1,
    comp_col_start=COMP_C0, comp_col_end=COMP_C1,
    atomLocations_buf=atomLocations_buf, imageProjs_buf=imageProjs_buf,
    imageProjs_local_buf=imageProjs_local_buf,
    imageProjs_local_size_buf=imageProjs_local_size_buf,
    fullImage_buf=fullImage_buf, emissions_buf=emissions_buf,
    targetGeometry_buf=targetGeometry_buf,
)
print(f"\u2713 {elapsed_ms:.1f} ms | moveCount={move_count} (expected 0)")

emissions_buf.invalidate()
emissions = np.copy(emissions_buf[:atom_count])

emin = float(np.min(emissions[emissions > 0])) if np.any(emissions > 0) else 0.0
emax = float(np.max(emissions))
EMISSION_THRESHOLD = (emax + emin) / 2.0   # adaptive, same as testbench

print(f"Emission range: [{emin:.1f}, {emax:.1f}]")
print(f"Adaptive threshold: {EMISSION_THRESHOLD:.1f}")

occupied = (emissions > EMISSION_THRESHOLD)
state_grid = occupied.reshape(GRID_R, GRID_C)
print(f"Occupied: {np.sum(occupied)}/{atom_count}  ({100*np.mean(occupied):.1f}%)")
print("Occupancy map (X=occupied):")
for r in range(GRID_R):
    row_str = "".join("X" if state_grid[r,c] else "." for c in range(GRID_C))
    print(f"  {row_str}")

## 7b. Run — MODE_INITIALIZATION

Runs image analysis + sorting. Moves streamed to AXI-Stream FIFO.

In [ ]:
# Reset FIFO before run
fifo.write(FIFO_RDFR, 0xA5)
time.sleep(0.01)

print("Running MODE_INITIALIZATION (draining FIFO concurrently)...")

# Write all registers
ctrl.write(MODE_OFFSET, MODE_INITIALIZATION)
write_float(ctrl, THRESHOLD_OFFSET, EMISSION_THRESHOLD)
ctrl.write(ATOM_LOC_SIZE, atom_count); ctrl.write(PROJ_SHAPE0, proj_shape0); ctrl.write(PROJ_SHAPE1, proj_shape1)
write_addr64(ctrl, ADDR_ATOM_LOC_LO, ADDR_ATOM_LOC_HI, atomLocations_buf.physical_address)
ctrl.write(PSF_SUPER, psf_super); ctrl.write(IMG_PROJ_SIZE, img_proj_sz)
write_addr64(ctrl, ADDR_PROJS_LOCAL_LO, ADDR_PROJS_LOCAL_HI, imageProjs_local_buf.physical_address)
write_addr64(ctrl, ADDR_IMAGE_PROJS_LO, ADDR_IMAGE_PROJS_HI, imageProjs_buf.physical_address)
write_addr64(ctrl, ADDR_PROJS_LOCAL_SZ_LO, ADDR_PROJS_LOCAL_SZ_HI, imageProjs_local_size_buf.physical_address)
write_addr64(ctrl, ADDR_FULL_IMAGE_LO, ADDR_FULL_IMAGE_HI, fullImage_buf.physical_address)
ctrl.write(FULL_IMG_ROWS, full_rows); ctrl.write(FULL_IMG_COLS, full_cols)
write_addr64(ctrl, ADDR_EMISSIONS_LO, ADDR_EMISSIONS_HI, emissions_buf.physical_address)
ctrl.write(GRID_ROWS, GRID_R); ctrl.write(GRID_COLS, GRID_C)
write_addr64(ctrl, ADDR_TARGET_GEO_LO, ADDR_TARGET_GEO_HI, targetGeometry_buf.physical_address)
ctrl.write(COMP_ROW_START, COMP_R0); ctrl.write(COMP_ROW_END, COMP_R1)
ctrl.write(COMP_COL_START, COMP_C0); ctrl.write(COMP_COL_END, COMP_C1)

rb_mode = ctrl.read(MODE_OFFSET) & 0xFF
rb_zrs  = ctrl.read(COMP_ROW_START)
rb_zre  = ctrl.read(COMP_ROW_END)
print(f"  [check] mode={rb_mode} compZoneRows=[{rb_zrs},{rb_zre})")

# Wait IDLE
for _ in range(10000):
    if ctrl.read(CTRL) & AP_IDLE:
        break
    time.sleep(0.001)

# Start IP
t0 = time.perf_counter()
ctrl.write(CTRL, AP_START)

# Poll AP_DONE AND drain FIFO simultaneously to prevent deadlock
# (FIFO has limited depth; if it fills up, IP stalls on moveStream.write)
all_raw_moves = []
t_first_move = None

while True:
    # Drain any available FIFO data
    while fifo.read(FIFO_RDFO) > 0:
        pkt_bytes = fifo.read(FIFO_RLR)
        words = (pkt_bytes + 3) // 4
        raw = bytearray()
        for _ in range(words):
            raw += struct.pack("<I", fifo.read(FIFO_RDFD))
        all_raw_moves.append(bytes(raw))
        if t_first_move is None:
            t_first_move = time.perf_counter()
            print(f"  First move at {(t_first_move - t0)*1e3:.1f} ms")

    if ctrl.read(CTRL) & AP_DONE:
        # Drain remaining FIFO after done
        while fifo.read(FIFO_RDFO) > 0:
            pkt_bytes = fifo.read(FIFO_RLR)
            words = (pkt_bytes + 3) // 4
            raw = bytearray()
            for _ in range(words):
                raw += struct.pack("<I", fifo.read(FIFO_RDFD))
            all_raw_moves.append(bytes(raw))
        break

    if time.perf_counter() - t0 > 60.0:
        print("Timeout!")
        break
    time.sleep(0.0001)

elapsed_ms = (time.perf_counter() - t0) * 1e3
move_count = ctrl.read(MOVE_COUNT_REG)
print(f"\u2713 {elapsed_ms:.1f} ms | moveCount={move_count} | FIFO packets read={len(all_raw_moves)}")

## 8. Read Emissions

In [ ]:
emissions_buf.invalidate()
emissions  = np.copy(emissions_buf[:atom_count])
occupied   = (emissions > EMISSION_THRESHOLD)
state_grid = occupied.reshape(GRID_R, GRID_C)
print(f"Occupied: {np.sum(occupied)}/{atom_count}  ({100*np.mean(occupied):.1f}%)")

plt.figure(figsize=(5, 5))
plt.imshow(state_grid.astype(int), cmap="Blues", origin="upper")
plt.title(f"Atom occupancy ({np.sum(occupied)} atoms)")
plt.colorbar(label="occupied")
plt.savefig("occupancy.png", dpi=150, bbox_inches="tight")
plt.show()

## 9. Read Moves from AXI-Stream FIFO

`ParallelMove` layout (must match `sortLatticeByRow.hpp`):
- `Step`: `colSelection[16]` int16 (32B) + `rowSelection[16]` int16 (32B) + counts (2B) = **66B**
- `ParallelMove`: `steps[20]` (1320B) + `stepsCount` uint8 = 1321B → padded to **1322B**
- Stream beats: ceil(1322/64) = **21 beats** = 1344B per move

In [ ]:
MAX_STEPS          = 20
MAX_SEL            = 16
STEP_SIZE          = 66
PARALLEL_MOVE_SIZE = 1322
BEATS_PER_MOVE     = 21
BYTES_PER_MOVE     = BEATS_PER_MOVE * 64

def decode_move(raw):
    """Decode ParallelMove from raw bytes."""
    if len(raw) < PARALLEL_MOVE_SIZE:
        return None
    steps_count = raw[MAX_STEPS * STEP_SIZE]
    if steps_count == 0 or steps_count > MAX_STEPS:
        return None
    steps = []
    for s in range(steps_count):
        off = s * STEP_SIZE
        cols_sel = list(struct.unpack_from(f"<{MAX_SEL}h", raw, off))
        rows_sel = list(struct.unpack_from(f"<{MAX_SEL}h", raw, off + 32))
        n_cols = raw[off + 64]
        n_rows = raw[off + 65]
        steps.append({"cols": cols_sel[:n_cols], "rows": rows_sel[:n_rows]})
    return steps

# Decode moves from all_raw_moves (already read from FIFO in cell 7b)
print(f"Decoding {len(all_raw_moves)} raw FIFO packets...")
all_moves = []
for i, raw in enumerate(all_raw_moves):
    move = decode_move(raw)
    if move is None:
        print(f"  Decode failed at move {i} (len={len(raw)})")
    else:
        all_moves.append(move)

print(f"\u2713 Decoded {len(all_moves)}/{move_count} moves")
if all_moves:
    spm = [len(m) for m in all_moves]
    print(f"  Steps/move: min={min(spm)}, max={max(spm)}, avg={np.mean(spm):.1f}")
    for i, m in enumerate(all_moves[:5]):
        print(f"  Move {i}: {len(m)} steps, first step rows={m[0]['rows'][:3]} cols={m[0]['cols'][:3]}")

## 10. Visualize Sorted State

In [ ]:
def apply_moves_to_grid(state, moves, grid_rows, grid_cols):
    result = state.copy().astype(bool)
    for move in moves:
        if not move:
            continue
        first, last = move[0], move[-1]
        moving = []
        for ri, r0 in enumerate(first["rows"]):
            for ci, c0 in enumerate(first["cols"]):
                if 0 <= r0 < grid_rows and 0 <= c0 < grid_cols and result[r0, c0]:
                    moving.append((ri, ci))
                    result[r0, c0] = False
        for ri, ci in moving:
            if ri < len(last["rows"]) and ci < len(last["cols"]):
                rN, cN = last["rows"][ri], last["cols"][ci]
                if 0 <= rN < grid_rows and 0 <= cN < grid_cols:
                    result[rN, cN] = True
    return result

sorted_grid = apply_moves_to_grid(state_grid, all_moves, GRID_R, GRID_C)

fig, axes = plt.subplots(1, 3, figsize=(14, 5))
axes[0].imshow(state_grid,  cmap="Blues",  origin="upper"); axes[0].set_title(f"Initial ({np.sum(state_grid)} atoms)");  axes[0].axis("off")
axes[1].imshow(sorted_grid, cmap="Blues",  origin="upper"); axes[1].set_title(f"After moves ({np.sum(sorted_grid)} atoms)"); axes[1].axis("off")
axes[2].imshow(target_2d,   cmap="Greens", origin="upper"); axes[2].set_title(f"Target ({np.sum(target_2d)} sites)"); axes[2].axis("off")
plt.tight_layout()
plt.savefig("atomflow_result.png", dpi=150, bbox_inches="tight")
plt.show()

zone_sorted = sorted_grid[COMP_R0:COMP_R1, COMP_C0:COMP_C1]
zone_target = target_2d[COMP_R0:COMP_R1, COMP_C0:COMP_C1].astype(bool)
diff = int(np.sum(zone_sorted != zone_target))
print(f"\n{'SUCCESS' if diff == 0 else f'WARNING: {diff} cells differ in comp zone'}")

## 11. Interleaved Pipeline Demo

Architecture (b): PS reads moves from FIFO while sorting is **still running**.
Shows lower latency to first move vs. waiting for AP_DONE.

In [ ]:
fifo.write(FIFO_RDFR, 0xA5)
time.sleep(0.01)

# Write all registers (same as cell 7b)
ctrl.write(MODE_OFFSET, MODE_INITIALIZATION)
write_float(ctrl, THRESHOLD_OFFSET, EMISSION_THRESHOLD)
ctrl.write(ATOM_LOC_SIZE, atom_count); ctrl.write(PROJ_SHAPE0, proj_shape0); ctrl.write(PROJ_SHAPE1, proj_shape1)
write_addr64(ctrl, ADDR_ATOM_LOC_LO, ADDR_ATOM_LOC_HI, atomLocations_buf.physical_address)
ctrl.write(PSF_SUPER, psf_super); ctrl.write(IMG_PROJ_SIZE, img_proj_sz)
write_addr64(ctrl, ADDR_PROJS_LOCAL_LO, ADDR_PROJS_LOCAL_HI, imageProjs_local_buf.physical_address)
write_addr64(ctrl, ADDR_IMAGE_PROJS_LO, ADDR_IMAGE_PROJS_HI, imageProjs_buf.physical_address)
write_addr64(ctrl, ADDR_PROJS_LOCAL_SZ_LO, ADDR_PROJS_LOCAL_SZ_HI, imageProjs_local_size_buf.physical_address)
write_addr64(ctrl, ADDR_FULL_IMAGE_LO, ADDR_FULL_IMAGE_HI, fullImage_buf.physical_address)
ctrl.write(FULL_IMG_ROWS, full_rows); ctrl.write(FULL_IMG_COLS, full_cols)
write_addr64(ctrl, ADDR_EMISSIONS_LO, ADDR_EMISSIONS_HI, emissions_buf.physical_address)
ctrl.write(GRID_ROWS, GRID_R); ctrl.write(GRID_COLS, GRID_C)
write_addr64(ctrl, ADDR_TARGET_GEO_LO, ADDR_TARGET_GEO_HI, targetGeometry_buf.physical_address)
ctrl.write(COMP_ROW_START, COMP_R0); ctrl.write(COMP_ROW_END, COMP_R1)
ctrl.write(COMP_COL_START, COMP_C0); ctrl.write(COMP_COL_END, COMP_C1)

while not (ctrl.read(CTRL) & AP_IDLE):
    time.sleep(0.001)

t_start = time.perf_counter()
ctrl.write(CTRL, AP_START)

# Read FIFO concurrently (interleaved pipeline demo)
t_first_move  = None
interleaved_moves = []
move_timestamps   = []

while True:
    done = bool(ctrl.read(CTRL) & AP_DONE)

    # Drain FIFO inline (no fifo_read_move dependency)
    while fifo.read(FIFO_RDFO) > 0:
        pkt_bytes = fifo.read(FIFO_RLR)
        words = (pkt_bytes + 3) // 4
        raw = bytearray()
        for _ in range(words):
            raw += struct.pack("<I", fifo.read(FIFO_RDFD))

        t_now = time.perf_counter()
        if t_first_move is None:
            t_first_move = t_now
            print(f"  First move: {(t_first_move - t_start)*1e3:.1f} ms after AP_START")

        move = decode_move(bytes(raw))
        if move:
            interleaved_moves.append(move)
            move_timestamps.append(t_now - t_start)

    if done:
        # Final drain
        while fifo.read(FIFO_RDFO) > 0:
            pkt_bytes = fifo.read(FIFO_RLR)
            words = (pkt_bytes + 3) // 4
            raw = bytearray()
            for _ in range(words):
                raw += struct.pack("<I", fifo.read(FIFO_RDFD))
            move = decode_move(bytes(raw))
            if move:
                interleaved_moves.append(move)
                move_timestamps.append(time.perf_counter() - t_start)
        break

    if time.perf_counter() - t_start > 60.0:
        print("Timeout!")
        break

t_total = (time.perf_counter() - t_start) * 1e3
t_first = (t_first_move - t_start) * 1e3 if t_first_move else float("nan")
mc = ctrl.read(MOVE_COUNT_REG)
print(f"\nInterleaved pipeline results:")
print(f"  AP_DONE:       {t_total:.1f} ms")
print(f"  First move:    {t_first:.1f} ms")
print(f"  Moves decoded: {len(interleaved_moves)} / {mc}")
if move_timestamps:
    print(f"  Timestamps:    {[f'{t*1e3:.1f}ms' for t in move_timestamps[:8]]}...")

## 12. Latency Benchmark

Measures per-stage latency for paper figures:
- **Image Analysis** (`reconstruct`): MODE_QUBIT_READOUT AP_START→AP_DONE
- **Sorting** (total + per-move): MODE_INITIALIZATION time − image analysis time
- **First-move latency**: time from AP_START to first FIFO packet (interleaved pipeline)

In [ ]:
# ---- Latency Benchmark ----
N_RUNS = 5  # number of repetitions for statistics

t_recon_list   = []   # MODE_QUBIT_READOUT (image analysis only)
t_total_list   = []   # MODE_INITIALIZATION (image analysis + sorting)
t_first_list   = []   # time to first move from FIFO
t_last_list    = []   # time to last move from FIFO
move_ts_all    = []   # per-move timestamps from best run
move_count_list = []

for run in range(N_RUNS):
    # --- 1) Image Analysis only (MODE_QUBIT_READOUT) ---
    for _ in range(10000):
        if ctrl.read(CTRL) & AP_IDLE: break
        time.sleep(0.001)
    ctrl.write(MODE_OFFSET, MODE_QUBIT_READOUT)
    write_float(ctrl, THRESHOLD_OFFSET, 0.0)
    ctrl.write(ATOM_LOC_SIZE, atom_count)
    ctrl.write(PROJ_SHAPE0, proj_shape0); ctrl.write(PROJ_SHAPE1, proj_shape1)
    write_addr64(ctrl, ADDR_ATOM_LOC_LO, ADDR_ATOM_LOC_HI, atomLocations_buf.physical_address)
    ctrl.write(PSF_SUPER, psf_super); ctrl.write(IMG_PROJ_SIZE, img_proj_sz)
    write_addr64(ctrl, ADDR_PROJS_LOCAL_LO, ADDR_PROJS_LOCAL_HI, imageProjs_local_buf.physical_address)
    write_addr64(ctrl, ADDR_IMAGE_PROJS_LO, ADDR_IMAGE_PROJS_HI, imageProjs_buf.physical_address)
    write_addr64(ctrl, ADDR_PROJS_LOCAL_SZ_LO, ADDR_PROJS_LOCAL_SZ_HI, imageProjs_local_size_buf.physical_address)
    write_addr64(ctrl, ADDR_FULL_IMAGE_LO, ADDR_FULL_IMAGE_HI, fullImage_buf.physical_address)
    ctrl.write(FULL_IMG_ROWS, full_rows); ctrl.write(FULL_IMG_COLS, full_cols)
    write_addr64(ctrl, ADDR_EMISSIONS_LO, ADDR_EMISSIONS_HI, emissions_buf.physical_address)
    ctrl.write(GRID_ROWS, GRID_R); ctrl.write(GRID_COLS, GRID_C)
    write_addr64(ctrl, ADDR_TARGET_GEO_LO, ADDR_TARGET_GEO_HI, targetGeometry_buf.physical_address)
    ctrl.write(COMP_ROW_START, COMP_R0); ctrl.write(COMP_ROW_END, COMP_R1)
    ctrl.write(COMP_COL_START, COMP_C0); ctrl.write(COMP_COL_END, COMP_C1)

    t0 = time.perf_counter()
    ctrl.write(CTRL, AP_START)
    while not (ctrl.read(CTRL) & AP_DONE):
        time.sleep(0.0001)
    t_recon = (time.perf_counter() - t0) * 1e3
    t_recon_list.append(t_recon)

    # --- 2) Full pipeline (MODE_INITIALIZATION) with FIFO draining ---
    fifo.write(FIFO_RDFR, 0xA5)
    time.sleep(0.01)

    for _ in range(10000):
        if ctrl.read(CTRL) & AP_IDLE: break
        time.sleep(0.001)
    ctrl.write(MODE_OFFSET, MODE_INITIALIZATION)
    write_float(ctrl, THRESHOLD_OFFSET, EMISSION_THRESHOLD)
    ctrl.write(ATOM_LOC_SIZE, atom_count)
    ctrl.write(PROJ_SHAPE0, proj_shape0); ctrl.write(PROJ_SHAPE1, proj_shape1)
    write_addr64(ctrl, ADDR_ATOM_LOC_LO, ADDR_ATOM_LOC_HI, atomLocations_buf.physical_address)
    ctrl.write(PSF_SUPER, psf_super); ctrl.write(IMG_PROJ_SIZE, img_proj_sz)
    write_addr64(ctrl, ADDR_PROJS_LOCAL_LO, ADDR_PROJS_LOCAL_HI, imageProjs_local_buf.physical_address)
    write_addr64(ctrl, ADDR_IMAGE_PROJS_LO, ADDR_IMAGE_PROJS_HI, imageProjs_buf.physical_address)
    write_addr64(ctrl, ADDR_PROJS_LOCAL_SZ_LO, ADDR_PROJS_LOCAL_SZ_HI, imageProjs_local_size_buf.physical_address)
    write_addr64(ctrl, ADDR_FULL_IMAGE_LO, ADDR_FULL_IMAGE_HI, fullImage_buf.physical_address)
    ctrl.write(FULL_IMG_ROWS, full_rows); ctrl.write(FULL_IMG_COLS, full_cols)
    write_addr64(ctrl, ADDR_EMISSIONS_LO, ADDR_EMISSIONS_HI, emissions_buf.physical_address)
    ctrl.write(GRID_ROWS, GRID_R); ctrl.write(GRID_COLS, GRID_C)
    write_addr64(ctrl, ADDR_TARGET_GEO_LO, ADDR_TARGET_GEO_HI, targetGeometry_buf.physical_address)
    ctrl.write(COMP_ROW_START, COMP_R0); ctrl.write(COMP_ROW_END, COMP_R1)
    ctrl.write(COMP_COL_START, COMP_C0); ctrl.write(COMP_COL_END, COMP_C1)

    raw_moves = []
    move_ts   = []
    t_first   = None

    t0 = time.perf_counter()
    ctrl.write(CTRL, AP_START)

    while True:
        while fifo.read(FIFO_RDFO) > 0:
            pkt_bytes = fifo.read(FIFO_RLR)
            words = (pkt_bytes + 3) // 4
            raw = bytearray()
            for _ in range(words):
                raw += struct.pack('<I', fifo.read(FIFO_RDFD))
            raw_moves.append(bytes(raw))
            t_now = time.perf_counter()
            move_ts.append((t_now - t0) * 1e3)
            if t_first is None:
                t_first = (t_now - t0) * 1e3

        if ctrl.read(CTRL) & AP_DONE:
            while fifo.read(FIFO_RDFO) > 0:
                pkt_bytes = fifo.read(FIFO_RLR)
                words = (pkt_bytes + 3) // 4
                raw = bytearray()
                for _ in range(words):
                    raw += struct.pack('<I', fifo.read(FIFO_RDFD))
                raw_moves.append(bytes(raw))
                move_ts.append((time.perf_counter() - t0) * 1e3)
            break

        if time.perf_counter() - t0 > 60.0:
            print(f'Run {run}: TIMEOUT'); break
        time.sleep(0.0001)

    t_total = (time.perf_counter() - t0) * 1e3
    mc = ctrl.read(MOVE_COUNT_REG)
    t_total_list.append(t_total)
    t_first_list.append(t_first if t_first else float('nan'))
    t_last_list.append(move_ts[-1] if move_ts else float('nan'))
    move_count_list.append(mc)
    if len(move_ts) > len(move_ts_all):
        move_ts_all = move_ts  # keep the run with most timestamps

    print(f'Run {run}: T_recon={t_recon:.1f}ms  T_total={t_total:.1f}ms  '
          f'T_first_move={t_first:.1f}ms  moves={mc}')

# --- Summary ---
t_recon_mean = np.mean(t_recon_list)
t_recon_std  = np.std(t_recon_list)
t_total_mean = np.mean(t_total_list)
t_total_std  = np.std(t_total_list)
t_sort_mean  = t_total_mean - t_recon_mean
t_first_mean = np.nanmean(t_first_list)

# Per-move interval (from timestamps)
if len(move_ts_all) > 1:
    intervals = np.diff(move_ts_all)
    t_per_move = np.mean(intervals)
else:
    t_per_move = float('nan')

print(f'\n{"="*50}')
print(f'Latency Breakdown ({N_RUNS} runs, 16x16 grid):')
print(f'{"="*50}')
print(f'  Image Analysis:     {t_recon_mean:7.1f} +/- {t_recon_std:.1f} ms')
print(f'  Sorting (total):    {t_sort_mean:7.1f} ms  (= T_total - T_recon)')
print(f'  Total (AP_DONE):    {t_total_mean:7.1f} +/- {t_total_std:.1f} ms')
print(f'  First move latency: {t_first_mean:7.1f} ms  (interleaved)')
print(f'  Per-move interval:  {t_per_move:7.2f} ms  (avg from timestamps)')
print(f'  Move count:         {move_count_list[-1]}')
print(f'{"="*50}')

In [ ]:
# ---- Stacked Bar Chart for Paper ----
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Left: stacked bar — total latency breakdown
ax = axes[0]
bars = ax.barh(['AtomFlow\nController'], [t_recon_mean], color='#4C72B0', label='Image Analysis')
ax.barh(['AtomFlow\nController'], [t_sort_mean], left=[t_recon_mean], color='#DD8452', label='Sorting')
ax.set_xlabel('Latency (ms)')
ax.set_title('End-to-End Latency Breakdown')
ax.legend(loc='lower right')
ax.set_xlim(0, t_total_mean * 1.15)
# Add text labels on bars
ax.text(t_recon_mean / 2, 0, f'{t_recon_mean:.1f} ms', ha='center', va='center', color='white', fontweight='bold')
ax.text(t_recon_mean + t_sort_mean / 2, 0, f'{t_sort_mean:.1f} ms', ha='center', va='center', color='white', fontweight='bold')

# Right: timeline — per-move arrival (interleaved pipeline)
ax2 = axes[1]
if move_ts_all:
    ax2.scatter(move_ts_all, range(len(move_ts_all)), s=15, c='#55A868', zorder=3)
    ax2.axvline(t_recon_mean, color='#4C72B0', ls='--', lw=1.5, label=f'Image Analysis done ({t_recon_mean:.0f} ms)')
    ax2.set_xlabel('Time since AP_START (ms)')
    ax2.set_ylabel('Move index')
    ax2.set_title('Move Arrival Timeline (Interleaved)')
    ax2.legend(fontsize=8)
    ax2.invert_yaxis()

plt.tight_layout()
plt.savefig('latency_breakdown.pdf', bbox_inches='tight', dpi=150)
plt.savefig('latency_breakdown.png', bbox_inches='tight', dpi=150)
plt.show()
print('Saved: latency_breakdown.pdf, latency_breakdown.png')

In [ ]:
# ---- Paper Table: Latency Breakdown ----

# Derived metrics
t_control = t_first_mean - t_recon_mean - t_per_move  # binary + control overhead
t_sort_total = t_per_move * move_count_list[-1]
mc = move_count_list[-1]

print('='*60)
print(f'  {"Stage":<32} {"Latency":>10} {"Source":>10}')
print('-'*60)
print(f'  {"Image Analysis":<32} {t_recon_mean:>8.1f} ms {"FPGA":>10}')
print(f'  {"Binary Class. + Control":<32} {t_control:>8.1f} ms {"derived":>10}')
print(f'  {"Sorting (per move)":<32} {t_per_move:>8.2f} ms {"FPGA":>10}')
print(f'  {"Sorting (" + str(mc) + " moves)":<32} {t_sort_total:>8.1f} ms {"FPGA":>10}')
print('-'*60)
print(f'  {"Total (end-to-end)":<32} {t_total_mean:>8.1f} ms {"FPGA":>10}')
print(f'  {"First-move latency":<32} {t_first_mean:>8.1f} ms {"FPGA":>10}')
print('='*60)

# --- LaTeX ---
pct_recon = 100 * t_recon_mean / t_total_mean
pct_ctrl  = 100 * t_control / t_total_mean
pct_sort  = 100 * t_sort_total / t_total_mean

latex = rf"""
\begin{{table}}[t]
\centering
\caption{{Latency breakdown of the AtomFlow controller on Zynq UltraScale+ ({mc} moves, 16$\times$16 grid).}}
\label{{tab:latency}}
\begin{{tabular}}{{@{{}}lrrr@{{}}}}
\toprule
Stage & Latency (ms) & \\% of Total & Source \\\\
\midrule
Image analysis          & {t_recon_mean:.1f} $\pm$ {t_recon_std:.1f} & {pct_recon:.1f}\\%  & FPGA \\\\
Binary classif. + ctrl  & {t_control:.1f}            & {pct_ctrl:.1f}\\%  & derived \\\\
Sorting ($\times${mc})  & {t_sort_total:.1f}           & {pct_sort:.1f}\\%  & FPGA \\\\
\quad per move          & {t_per_move:.2f}           & ---          & FPGA \\\\
\midrule
\textbf{{Total}}        & \textbf{{{t_total_mean:.1f}}} $\pm$ {t_total_std:.1f} & 100\\% & FPGA \\\\
\midrule
First-move latency      & {t_first_mean:.1f}            & ---          & FPGA \\\\
\bottomrule
\end{{tabular}}
\end{{table}}
"""
print(latex)


## 13. Cleanup

In [ ]:
print("Done.")